# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/metrosmash/myflyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- One row = One unique piece of content (content_id) for a specific client. over a 90 days time window
this allows for us to give the page 30 days of comparision time and a previous 30 days check with a last 30 days check


- each page_position needs to be grouped.

- we have to setup a Definition of "Underperforming": The guide highlights that CTR is strongly determined by a page's SERP Position. A CTR of 1% is terrible for Position 1, but amazing for Position 9. Instead of using a hard cutoff for clicks (clicks <= 7), the guide suggests comparing a page's actual CTR against the average expected CTR for its SERP Position (using the position_tier or avg_position).

- Reason Codes: The lane guidelines recommend attaching explicit reasons for the flagging (e.g., "good_position_bad_ctr" or "stale_and_declining") to make the scores actionable for content writers.

In [20]:
# installing needed libriaries
%pip -q install duckdb huggingface_hub

# importing libraries
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import os, sys, subprocess
import os, getpass
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('Hf_read_token')

In [21]:
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/metrosmash/myflyrank-ml-internship"
REPO_DIR = "myflyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")



print("Loading the data ")
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Data Loaded !!!")

Working dir: /content/myflyrank-ml-internship/myflyrank-ml-internship
Starter data found. You're ready.
Loading the data 
Data Loaded !!!


In [22]:
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')

dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


In [10]:
df['position_tier'].value_counts()

,count
position_tier,
page_1,11814
striking,7304
page_3_5,7242
top_3,2321
deep,1319


In [14]:
# 1. Verify Grain: client_id x content_id
grain_cols = ['client_id', 'content_id']
duplicates = df.duplicated(subset=grain_cols).sum()
unique_combinations = df[grain_cols].drop_duplicates().shape[0]
print(f"--- Grain Verification ---")
print(f"Unique client_id x content_id pairs: {unique_combinations}")
print(f"Duplicate rows on grain: {duplicates}")

# 2. Filter out rows with avg_position = 0 (representing 'no data')
valid_pos_df = df[df['avg_position'] > 0].copy()

# 3. Calculate expected baseline CTR (%) by existing position_tier
print("\n--- Expected CTR (%) by Position Tier ---")
# Recall: rate columns in this dataset are x100 percentages (e.g., 0.76 means 0.76%)
ctr_by_tier = valid_pos_df.groupby('position_tier', observed=False)['ctr'].mean().reset_index()

# Sort for logical presentation if possible
display(ctr_by_tier.sort_values('ctr', ascending=False))

--- Grain Verification ---
Unique client_id x content_id pairs: 30000
Duplicate rows on grain: 0

--- Expected CTR (%) by Position Tier ---


,position_tier,ctr
4,top_3,2.764453
1,page_1,0.652467
3,striking,0.323239
2,page_3_5,0.222484
0,deep,0.150212


## 2. Fields: feature / label / context / excluded

### Feature
- **Static Metadata**: `search_volume`, `competition`, `word_count` (provides baseline page attributes).
- **SERP Position Metrics**: `avg_position`, `position_tier` (crucial non-linear context needed to evaluate baseline expected CTRs).
- **Historical Engagement Window**: `prev_30_days_clicks`, `prev_30_days_impressions`, `prev_30_days_ctr` (safely captures pre-outcome traffic and performance without leakage).

### label
- **Primary engineered label (`is_underperforming`)**: A binary indicator ($0$ or $1$) defined as: `last_30_days_ctr < expected_ctr_for_tier` (where `expected_ctr_for_tier` is the average CTR of the page's respective `position_tier`).
- **Auxiliary pipeline target (`trend_direction`)**: Multi-class label representing organic performance state (`stable`, `down`, `up`).

### context
- **Identifiers**: `client_id` (used for client-holdout validation splits to test model generalization), `content_id` (defines the unique grain, used for grouping/joins only).
- **Descriptive Columns**: `url`, `keyword` (retained for mapping, analysis, and human-in-the-loop actionability; excluded from modeling to prevent overfitting on raw text strings).

### excluded
- **Target Leakage Risks**: `last_30_days_clicks`, `last_30_days_impressions`, and `last_30_days_ctr` are strictly excluded from being model features because they represent current performance inside our sealed outcome window. They are only utilized mathematically to define the true labels.
- **Direct Target Proxies**: `trend_pct` and raw `trend_direction` are excluded from features as they directly encode the outcome state.

### 1. Unified Categorization Table

To ensure our model is mathematically sound, free from target leakage, and highly compliant with client privacy policies, we categorize the available schema columns as follows:

| Column Name | Category | Role & Justification | Prevents Leakage & Protects Privacy? |
| :--- | :--- | :--- | :--- |
| `content_id`, `keyword` | **Context** | Identifiers used for mapping, joining, and human-in-the-loop reporting. Model training ignores these to avoid memorizing high-cardinality values. | Yes |
| `search_volume`, `competition`, `word_count` | **Feature** | Static demand, competition tier, and structural quality characteristics of the content grain. | Yes |
| `position_tier`, `avg_position` | **Feature** | High-signal position baselines. Used to set the objective expected CTR threshold. | Yes |
| `prev_30_days_clicks`, `prev_30_days_impressions`, `prev_30_days_ctr` | **Feature** | Historical baseline engagement metrics. This acts as the feature baseline window. | Yes |
| `last_30_days_clicks`, `last_30_days_impressions`, `last_30_days_ctr` | **Excluded (as feature) / Source of Label** | These contain outcome window metrics. Using them directly as features guarantees catastrophic target leakage. | Yes (Isolated to Label Creation) |
| `is_underperforming` | **Label (Primary)** | Binary classification target. Equal to `1` if `last_30_days_ctr < expected_ctr_for_tier`, else `0`. | Target Variable |
| `performance_trend` | **Label (Auxiliary)** | Multi-class outcome category (`stable`, `down`, `up`) indicating direction of change. | Target Variable |
| `client_id`, `url` | **Excluded (Strict Privacy Rule)** | **Excluded completely from all model inputs.** Using raw URLs or client IDs poses severe data security and privacy risks when serving shared models. | **Yes (Enforces Client Anonymity)** |

---

### 2. Operational Definitions

* **Features**: Static parameters (`search_volume`, `competition`, `word_count`), baseline position indicators (`position_tier`), and metrics from the historical window (`prev_30_days_clicks`, `prev_30_days_impressions`, `prev_30_days_ctr`).
* **Labels**:
  * Primary: `is_underperforming` (engineered binary flag using outcome-window CTR comparison to position tier baselines).
  * Secondary: `performance_trend` (directly available in the dataset).
* **Context**: Metadata such as `content_id` and `keyword` to ensure readability and downstream actions.
* **Excluded (Privacy & Leakage Policy)**:
  * **Leakage Avoidance**: All variables directly summarizing the outcome window (`last_30_days_*`) are excluded from model inputs.
  * **Privacy Protection**: `client_id` and raw `url` columns are strictly excluded. The warehouse only exposes anonymous hash identifiers (`client_hash_id`, `content_hash_id`) to ensure client privacy is preserved across the engineering pipeline.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

### 3. Verification Queries & Statistical Proofs (Results & Interpretation)

We ran programmatic verification checks directly against the full warehouse sample (`fact_content_daily_performance_sample` via DuckDB) to stress-test our data contract assumptions. Here are the empirical findings:

#### 1. Grain Uniqueness Check
* **Expected behavior**: We checked whether `client_hash_id` + `content_hash_id` + `report_date` is unique in the raw daily table.
* **Observed reality**: The query returned duplicate records (e.g., matching the same key triplet on `2026-06-13` with a count of `2`). This indicates that raw daily data can have multiple entries for the same content piece (e.g., due to multiple protocol variants, tracking parameters, or device splits that have not been collapsed).
* **Actionable Rule**: To maintain a strictly unique unit of analysis, our data pipeline must explicitly group and sum/average metrics (using `SUM(gsc_clicks)`, `SUM(gsc_impressions)`, and weighted `gsc_avg_position`) across our defined 90-day aggregation window rather than assuming raw daily rows are unique.

#### 2. Missing Values & Zero Position Check
* **Observed reality**: Out of $11,694,072$ records, **$7,885,413$ ($67.43\%$)** contain null or zero average positions.
* **Actionable Rule**: This confirms that the majority of tracked content does not rank or generate impressions on any given day. To avoid dividing by zero or introducing severe skewness, our label engineering must discard these unranked days or apply Laplace smoothing before evaluating performance.

#### 3. Expected CTR Baseline
* **Observed baseline distribution**:
  - **Pos 1**: `8.5690%` CTR
  - **Pos 2-3**: `1.7956%` CTR
  - **Pos 4-6**: `0.5070%` CTR
  - **Page 2+**: `0.3523%` CTR
  - **Pos 7-10**: `0.2925%` CTR

This validates that CTR drops exponentially with SERP positions, confirming that any performance label (`is_underperforming`) must evaluate actual CTR relative to these non-linear thresholds rather than using a hard threshold.

In [29]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

fact_daily_src = TABLES['fact_daily']
fact_daily_sample_src = TABLES['fact_daily_sample']

# 1. Programmatically verify grain uniqueness in warehouse daily performance
# The grain is client_hash_id x content_hash_id per report_date
double_grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*)
    FROM {fact_daily_sample_src}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").fetchall()
print("Grain Check (Duplicates per client_hash_id x content_hash_id x report_date in sample):", double_grain_check)

# 2. Check for Missing Values & Zero/Negative Positions in the warehouse sample
zero_pos_stats = con.sql(f"""
    SELECT
        COUNT(*) as total_records,
        SUM(CASE WHEN gsc_avg_position IS NULL OR gsc_avg_position <= 0 THEN 1 ELSE 0 END) as zero_or_null_positions,
        ROUND(100.0 * SUM(CASE WHEN gsc_avg_position IS NULL OR gsc_avg_position <= 0 THEN 1 ELSE 0 END) / COUNT(*), 2) as zero_pos_pct
    FROM {fact_daily_sample_src}
""").df()
print("\n--- Position Zero/Null Check in Warehouse Sample ---")
display(zero_pos_stats)

# 3. Calculate baseline CTR by Position Buckets on the warehouse sample dataset
# Note: gsc_clicks / gsc_impressions provides actual CTR. We handle null/zero impressions safely.
tier_baseline = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position BETWEEN 1 AND 1.5 THEN 'Pos 1'
            WHEN gsc_avg_position BETWEEN 1.5 AND 3.5 THEN 'Pos 2-3'
            WHEN gsc_avg_position BETWEEN 3.5 AND 6.5 THEN 'Pos 4-6'
            WHEN gsc_avg_position BETWEEN 6.5 AND 10.5 THEN 'Pos 7-10'
            ELSE 'Page 2+'
        END AS simple_tier,
        ROUND(SUM(gsc_clicks) * 100.0 / NULLIF(SUM(gsc_impressions), 0), 4) as avg_ctr_pct
    FROM {fact_daily_sample_src}
    WHERE gsc_avg_position > 0 AND gsc_impressions > 0
    GROUP BY simple_tier
    ORDER BY avg_ctr_pct DESC
""").df()
print("\n--- Expected CTR (%) by Simple Tier (Warehouse Sample Summary) ---")
display(tier_baseline)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain Check (Duplicates per client_hash_id x content_hash_id x report_date in sample): [('client_e00b29e582949543', 'content_8e51f9165149270e', datetime.date(2026, 6, 13), 2), ('client_e00b29e582949543', 'content_81c1fc0cd0d023b9', datetime.date(2026, 6, 13), 2), ('client_e00b29e582949543', 'content_638ac5ad8023d177', datetime.date(2026, 6, 13), 2), ('client_e00b29e582949543', 'content_ac3831ae8497a17e', datetime.date(2026, 6, 13), 2), ('client_e00b29e582949543', 'content_c58c118161c27095', datetime.date(2026, 6, 13), 2)]


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Position Zero/Null Check in Warehouse Sample ---


,total_records,zero_or_null_positions,zero_pos_pct
0,11694072,7885413.0,67.43


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- Expected CTR (%) by Simple Tier (Warehouse Sample Summary) ---


,simple_tier,avg_ctr_pct
0,Pos 1,8.5690
1,Pos 2-3,1.7956
2,Pos 4-6,0.5070
3,Page 2+,0.3523
4,Pos 7-10,0.2925


## 4. Data limits & What This Data Can Never Tell You

Based on empirical query diagnostics conducted directly on the warehouse sample, we identify three critical, structural blind spots where the data is fundamentally limited:

### 1. Unbalanced History (Varying Time Windows)
* **Observed Reality**: The duration of tracking spans differs between clients. While some accounts have a full 29-day tracking window (e.g., `client_59256b0571e0c970` with 29 history days), others have only 24 days of history (e.g., `client_2c32078d69f2cbad` ending on `2026-06-25`).
* **Limit**: We can never assume that aggregating history across a uniform global threshold (like 90 days) will yield equivalent statistical support or equal sample sizes across all clients in production.

### 2. GSC vs GA4 Integration Asymmetry
* **Observed Reality**: **20.25%** of all performance records ($2,368,462$ out of $11,694,072$) have search impressions and clicks (`gsc_data_available = TRUE`) but absolutely no downstream analytics integration (`ga4_data_available = FALSE`).
* **Limit**: For more than one-fifth of our dataset, the data can never tell us what users do after clicking the link (e.g., page engagement, bounce rates, scrolling, or direct conversions). Models utilizing GA4 signals as features will suffer from systematic missingness or bias if applied broadly.

### 3. Latent User Intent & Search Algorithm Changes
* **Limit**: The dataset tracks quantitative behaviors (clicks, impressions, position). However, it can never capture the *why*—such as changes in search engine presentation (SERP feature adjustments, AI Overviews), direct competitor changes, or sudden user behavior trends that do not map directly to historical numbers.

### 4. Data Limits & What This Data Can Never Tell You

Every dataset has structural blind spots. Rather than guessing, we run diagnostics below on the warehouse sample to quantify three core limitations:
1. **Unbalanced History**: Do all clients have the same history window length?
2. **GSC vs GA4 Asymmetry**: How often are we blind to user behavior (GA4) despite having search visibility (GSC)?
3. **Window Gaps**: Do we have continuous tracking data, or are there silent reporting gaps?

In [30]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

fact_daily_sample_src = TABLES['fact_daily_sample']

# 1. Investigate historical window span by client
client_history = con.sql(f"""
    SELECT
        client_hash_id,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date,
        DATEDIFF('day', MIN(report_date), MAX(report_date)) as history_span_days
    FROM {fact_daily_sample_src}
    GROUP BY client_hash_id
    ORDER BY history_span_days DESC
""").df()

print("--- History Window Span (Top 5 clients with longest vs shortest spans) ---")
display(client_history.head(5))
display(client_history.tail(5))

# 2. Investigate GSC-only records (asymmetry where GA4 post-click user action is missing)
ga4_vs_gsc = con.sql(f"""
    SELECT
        COUNT(*) as total_records,
        SUM(CASE WHEN gsc_data_available = TRUE AND ga4_data_available = FALSE THEN 1 ELSE 0 END) as gsc_only_records,
        ROUND(100.0 * SUM(CASE WHEN gsc_data_available = TRUE AND ga4_data_available = FALSE THEN 1 ELSE 0 END) / COUNT(*), 2) as gsc_only_pct
    FROM {fact_daily_sample_src}
""").df()

print("\n--- GSC vs GA4 Integration Asymmetry ---")
display(ga4_vs_gsc)

--- History Window Span (Top 5 clients with longest vs shortest spans) ---


,client_hash_id,start_date,end_date,history_span_days
0,client_59256b0571e0c970,2026-06-01,2026-06-30,29
1,client_a1203ffecad62470,2026-06-01,2026-06-30,29
2,client_795153d5b7850ccf,2026-06-01,2026-06-30,29
3,client_599043c0ff13edea,2026-06-01,2026-06-30,29
4,client_770e8e5faa9cddfe,2026-06-01,2026-06-30,29


,client_hash_id,start_date,end_date,history_span_days
60,client_cd12bcfd98942aa1,2026-06-01,2026-06-27,26
61,client_0797ff3a1fc9a6a5,2026-06-01,2026-06-25,24
62,client_f623b01661d4bfe4,2026-06-01,2026-06-25,24
63,client_19b89ee4fe3db6da,2026-06-01,2026-06-25,24
64,client_2c32078d69f2cbad,2026-06-01,2026-06-25,24


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


--- GSC vs GA4 Integration Asymmetry ---


,total_records,gsc_only_records,gsc_only_pct
0,11694072,2368462.0,20.25


## Self-check

Before you submit, confirm each line honestly:

- [*] Every section above is filled — markdown thinking AND the code that backs it
- [*] The notebook runs top to bottom with no errors (Runtime → Run all)
- [*] No client names, URLs, or private queries anywhere
- [*] My claims use careful words: observed, measured, directional, decision-support
- [*] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.